---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # ORDER_CHANNEL_SPLIT (converted from Dataiku DSS)

    | DSS recipe | Lakeflow Designer |
    |---|---|
    | compute_orders_prepared (Prepare) | `prep_filter_status`, `prep_derive` |
    | split_orders_by_channel (Split, VALUES on channel, default + missing -> orders_other) | `split_web`, `split_store`, `split_other` |
    | stack_web_store (Stack, UNION, origin column) | `stack_origin_web`, `stack_origin_store`, `stack_union` |

    Stack unions **by name**; Combine unions **by position**, so both branches select the same columns in the same order first.
---

In [0]:
"""
id: "src_orders"
template: "source"
templateVersion: "1.0.0"
name: "src_orders"
position:
  x: 0
  y: 360
description:
  text: "Dataset orders."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_orders"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_orders.data"] = out["data"]


In [0]:
"""
id: "prep_filter_status"
template: "filter"
templateVersion: "1.0.0"
name: "prep_filter_status"
position:
  x: 300
  y: 360
description:
  text: "Prepare step 1: FilterOnValue status in [COMPLETED, RETURNED] (KEEP_ROW)."
  hash: ""
previewMode: "1000"
config:
  condition: "status IN ('COMPLETED', 'RETURNED')"
input:
  - node: "src_orders"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status IN ('COMPLETED', 'RETURNED')"
}
inputs = {
    "data": ctx["src_orders.data"]
}
out = run(config, inputs, spark)
ctx["prep_filter_status.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "prep_derive"
template: "transform"
templateVersion: "1.0.0"
name: "prep_derive"
position:
  x: 600
  y: 360
description:
  text: "Prepare steps 2-4: formulas order_quarter (datePart month), is_return; ColumnsSelector removes store_id."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "order_id"
    - "customer_id"
    - "order_date"
    - "channel"
    - "status"
    - "CASE WHEN MONTH(order_date) <= 3 THEN 'Q1' WHEN MONTH(order_date) <= 6 THEN 'Q2' WHEN MONTH(order_date) <= 9 THEN 'Q3' ELSE 'Q4' END AS order_quarter"
    - "CASE WHEN status = 'RETURNED' THEN 1 ELSE 0 END AS is_return"
input:
  - node: "prep_filter_status"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_id",
        "customer_id",
        "order_date",
        "channel",
        "status",
        "CASE WHEN MONTH(order_date) <= 3 THEN 'Q1' WHEN MONTH(order_date) <= 6 THEN 'Q2' WHEN MONTH(order_date) <= 9 THEN 'Q3' ELSE 'Q4' END AS order_quarter",
        "CASE WHEN status = 'RETURNED' THEN 1 ELSE 0 END AS is_return"
    ]
}
inputs = {
    "data": ctx["prep_filter_status.filtered_data"]
}
out = run(config, inputs, spark)
ctx["prep_derive.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "split_web"
template: "filter"
templateVersion: "1.0.0"
name: "split_web"
position:
  x: 900
  y: 360
description:
  text: "Split value WEB -> orders_web."
  hash: ""
previewMode: "1000"
config:
  condition: "channel = 'WEB'"
input:
  - node: "prep_derive"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "channel = 'WEB'"
}
inputs = {
    "data": ctx["prep_derive.transformed_data"]
}
out = run(config, inputs, spark)
ctx["split_web.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "split_store"
template: "filter"
templateVersion: "1.0.0"
name: "split_store"
position:
  x: 900
  y: 515
description:
  text: "Split value STORE -> orders_store."
  hash: ""
previewMode: "1000"
config:
  condition: "channel = 'STORE'"
input:
  - node: "prep_derive"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "channel = 'STORE'"
}
inputs = {
    "data": ctx["prep_derive.transformed_data"]
}
out = run(config, inputs, spark)
ctx["split_store.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "split_other"
template: "filter"
templateVersion: "1.0.0"
name: "split_other"
position:
  x: 900
  y: 670
description:
  text: "Split default + missing values -> orders_other."
  hash: ""
previewMode: "1000"
config:
  condition: "channel IS NULL OR channel NOT IN ('WEB', 'STORE')"
input:
  - node: "prep_derive"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "channel IS NULL OR channel NOT IN ('WEB', 'STORE')"
}
inputs = {
    "data": ctx["prep_derive.transformed_data"]
}
out = run(config, inputs, spark)
ctx["split_other.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "stack_origin_web"
template: "transform"
templateVersion: "1.0.0"
name: "stack_origin_web"
position:
  x: 1200
  y: 360
description:
  text: "Stack origin column for orders_web."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "order_id"
    - "customer_id"
    - "order_date"
    - "channel"
    - "status"
    - "order_quarter"
    - "is_return"
    - "'orders_web' AS original_dataset"
input:
  - node: "split_web"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_id",
        "customer_id",
        "order_date",
        "channel",
        "status",
        "order_quarter",
        "is_return",
        "'orders_web' AS original_dataset"
    ]
}
inputs = {
    "data": ctx["split_web.filtered_data"]
}
out = run(config, inputs, spark)
ctx["stack_origin_web.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "stack_origin_store"
template: "transform"
templateVersion: "1.0.0"
name: "stack_origin_store"
position:
  x: 1200
  y: 515
description:
  text: "Stack origin column for orders_store."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "order_id"
    - "customer_id"
    - "order_date"
    - "channel"
    - "status"
    - "order_quarter"
    - "is_return"
    - "'orders_store' AS original_dataset"
input:
  - node: "split_store"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_id",
        "customer_id",
        "order_date",
        "channel",
        "status",
        "order_quarter",
        "is_return",
        "'orders_store' AS original_dataset"
    ]
}
inputs = {
    "data": ctx["split_store.filtered_data"]
}
out = run(config, inputs, spark)
ctx["stack_origin_store.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "stack_union"
template: "combine"
name: "stack_union"
position:
  x: 1500
  y: 360
description:
  text: "Stack UNION (columns aligned by name upstream)."
  hash: ""
previewMode: "1000"
config:
  operator: "UNION"
  quantifier: "ALL"
input:
  - node: "stack_origin_web"
    input_port: "data_0"
    output_port: "transformed_data"
  - node: "stack_origin_store"
    input_port: "data_1"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df_0 = inputs["data_0"]
    df_1 = inputs["data_1"]
    operator = config.get("operator", "UNION")
    quantifier = config.get("quantifier", "DISTINCT")

    op_map = {
        "UNION": lambda a, b: a.union(b),
        "INTERSECT": lambda a, b: a.intersectAll(b),
        "EXCEPT": lambda a, b: a.exceptAll(b),
        "MINUS": lambda a, b: a.exceptAll(b),
    }
    op_distinct_map = {
        "UNION": lambda a, b: a.union(b).distinct(),
        "INTERSECT": lambda a, b: a.intersect(b),
        "EXCEPT": lambda a, b: a.subtract(b),
        "MINUS": lambda a, b: a.subtract(b),
    }
    if quantifier == "DISTINCT":
        combine_fn = op_distinct_map.get(operator)
    else:
        combine_fn = op_map.get(operator)
    if not combine_fn:
        raise ValueError(f"Unsupported combine operator: {operator}")

    return {"combined_data": combine_fn(df_0, df_1)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "operator": "UNION",
    "quantifier": "ALL"
}
inputs = {
    "data_0": ctx["stack_origin_web.transformed_data"],
    "data_1": ctx["stack_origin_store.transformed_data"]
}
out = run(config, inputs, spark)
ctx["stack_union.combined_data"] = out["combined_data"]


In [0]:
"""
id: "out_orders_web_store"
template: "output"
templateVersion: "1.0.0"
name: "out_orders_web_store"
position:
  x: 1800
  y: 360
description:
  text: "Managed dataset orders_web_store."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "dss_03_orders_web_store"
input:
  - node: "stack_union"
    input_port: "data"
    output_port: "combined_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "dss_03_orders_web_store"
}
inputs = {
    "data": ctx["stack_union.combined_data"]
}
out = run(config, inputs, spark)


In [0]:
"""
id: "out_orders_other"
template: "output"
templateVersion: "1.0.0"
name: "out_orders_other"
position:
  x: 1200
  y: 670
description:
  text: "Managed dataset orders_other."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "dss_03_orders_other"
input:
  - node: "split_other"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "dss_03_orders_other"
}
inputs = {
    "data": ctx["split_other.filtered_data"]
}
out = run(config, inputs, spark)
